# SchoolBridge Fine-Tuning with Unsloth

Fine-tune Gemma 4 E4B for school notice extraction using QLoRA.

**Requirements**: Run on Kaggle with GPU T4 x2 or P100 accelerator enabled.

In [ ]:
%%capture
!pip install unsloth
!pip install --upgrade --no-cache-dir --no-deps unsloth unsloth_zoo
!pip install --upgrade transformers datasets trl accelerate peft bitsandbytes sentencepiece protobuf

In [ ]:
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name="unsloth/gemma-4-E4B-it",
    dtype=None,
    max_seq_length=2048,
    load_in_4bit=True,
    full_finetuning=False,
)

print("Model loaded successfully!")
print(f"Model type: {type(model).__name__}")

In [ ]:
model = FastModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

model.print_trainable_parameters()

In [ ]:
import os, glob
from datasets import load_dataset

# Try standard path first, then discover dataset location
data_dir = "/kaggle/input/schoolbridge-training-data"
train_file = os.path.join(data_dir, "train.jsonl")

if not os.path.exists(train_file):
    print(f"Dataset not found at {data_dir}, searching /kaggle/input/...")
    input_base = "/kaggle/input"
    if os.path.exists(input_base):
        for d in os.listdir(input_base):
            full = os.path.join(input_base, d)
            files = os.listdir(full) if os.path.isdir(full) else []
            print(f"  {full}/: {files}")
            if "train.jsonl" in files:
                data_dir = full
                train_file = os.path.join(full, "train.jsonl")
                print(f"  -> Found dataset at: {data_dir}")
                break
    else:
        print(f"  /kaggle/input does not exist!")

if not os.path.exists(train_file):
    print("Dataset not mounted — downloading directly...")
    os.system("kaggle datasets download rohanpatnaik/schoolbridge-training-data -p /tmp/sb-data --unzip")
    if os.path.exists("/tmp/sb-data/train.jsonl"):
        data_dir = "/tmp/sb-data"
        train_file = "/tmp/sb-data/train.jsonl"
    else:
        print("Contents of /tmp/sb-data:", os.listdir("/tmp/sb-data") if os.path.exists("/tmp/sb-data") else "NOT FOUND")
        raise FileNotFoundError("Could not find or download training data!")

val_file = os.path.join(data_dir, "val.jsonl")
print(f"\nUsing data_dir: {data_dir}")
print(f"Train file exists: {os.path.exists(train_file)}")
print(f"Val file exists: {os.path.exists(val_file)}")

dataset = load_dataset("json", data_files={
    "train": train_file,
    "validation": val_file,
})

print(f"\nTrain: {len(dataset['train'])} examples")
print(f"Validation: {len(dataset['validation'])} examples")
print(f"Sample keys: {list(dataset['train'][0].keys())}")
print(f"Sample conversation roles: {[c['role'] for c in dataset['train'][0]['conversations']]}")

In [ ]:
# Format conversations using the tokenizer's built-in chat template
def apply_template(examples):
    texts = []
    for convo in examples["conversations"]:
        messages = []
        for turn in convo:
            role = turn["role"]
            if role == "assistant":
                role = "model"
            messages.append({"role": role, "content": turn["content"]})
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(apply_template, batched=True)

print("Sample formatted text (first 600 chars):")
print(dataset["train"][0]["text"][:600])
print("\n...")
print(f"\nTotal length: {len(dataset['train'][0]['text'])} chars")

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

sft_config = SFTConfig(
    output_dir="./outputs",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    eval_strategy="no",
    save_strategy="no",
    seed=42,
    report_to="none",
    dataset_text_field="text",
    max_seq_length=2048,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    args=sft_config,
)

# Detect the correct turn markers from formatted text
sample_text = dataset["train"][0]["text"]
if "<start_of_turn>" in sample_text:
    user_part = "<start_of_turn>user\n"
    model_part = "<start_of_turn>model\n"
elif "<|turn>" in sample_text:
    user_part = "<|turn>user\n"
    model_part = "<|turn>model\n"
else:
    print(f"WARNING: Unknown chat template format. Sample: {sample_text[:200]}")
    user_part = "<|turn>user\n"
    model_part = "<|turn>model\n"

print(f"Using turn markers: user='{user_part.strip()}', model='{model_part.strip()}'")

trainer = train_on_responses_only(
    trainer,
    instruction_part=user_part,
    response_part=model_part,
)

print("Trainer ready!")

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
total_mem = getattr(gpu_stats, 'total_memory', None) or getattr(gpu_stats, 'total_mem', 0)
print(f"GPU: {gpu_stats.name} ({total_mem / 1024**3:.1f} GB)")
print(f"CUDA: {torch.version.cuda}")
print(f"\nStarting training...\n")

stats = trainer.train()

print(f"\n{'='*50}")
print(f"Training complete!")
print(f"Final train loss: {stats.training_loss:.4f}")
print(f"Runtime: {stats.metrics['train_runtime']:.0f}s ({stats.metrics['train_runtime']/60:.1f} min)")

In [ ]:
# Clear memory before inference
import gc
gc.collect()
torch.cuda.empty_cache()

from transformers import TextStreamer

test_notice = """Dear Parent, Your child has 8 absences this semester exceeding the 5 absence threshold.
Please contact the attendance office within 5 days. State law requires reporting excessive absences.
If medical, provide healthcare documentation. - Lincoln Elementary"""

messages = [
    {"role": "user", "content": f"Analyze this school notice:\n\n{test_notice}\n\nTarget language: English"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to("cuda")

print("Model output:")
_ = model.generate(
    **inputs,
    max_new_tokens=1024,
    temperature=0.1,
    do_sample=True,
    streamer=TextStreamer(tokenizer, skip_prompt=True),
)

In [ ]:
# Save LoRA adapters (small, fits on Kaggle disk)
# GGUF conversion will be done locally after download
import gc, os
gc.collect()
torch.cuda.empty_cache()

# Save just the LoRA adapter weights
model.save_pretrained("schoolbridge-lora-adapter")
tokenizer.save_pretrained("schoolbridge-lora-adapter")

print("LoRA adapter saved!")
for f in os.listdir("schoolbridge-lora-adapter"):
    size_mb = os.path.getsize(os.path.join("schoolbridge-lora-adapter", f)) / 1024 / 1024
    print(f"  {f} ({size_mb:.1f} MB)")

print("\nNext steps:")
print("1. Download adapter from the Output tab")
print("2. Merge with base model locally")
print("3. Export to GGUF for Ollama")